# ScrubRef: corrected, reviewer-ready, RAM-safe experiment notebook

This notebook replaces the original cell-dependent Google Colab workflow with a deterministic experiment pipeline for class-level machine unlearning.

It provides two explicitly separated ScrubRef variants:

1. **`scrubref_head`** — the corrected version of the original classifier-head method. It fixes the zero-gradient scrub branch, output/logit mismatch, missing distillation, missing MixUp, and reproducibility problems.
2. **`scrubref_rep`** — a reviewer-strengthened extension that also updates a small tail of the backbone and applies representation-level proxy alignment while preserving retained representations.

The notebook also includes:

- genuine membership-inference evaluation using forgotten **training members** versus forgotten **test non-members** from the same classes;
- retained utility and calibration metrics;
- retraining, logit-clamp, retained fine-tuning, head-reset, UNSIR-style, Bad Teacher, SCRUB-style, and SSD baselines;
- linear-probe, k-nearest-neighbor, and CKA representation audits;
- parameter-change, runtime, and peak-memory measurements;
- deterministic multi-seed execution;
- CSV, JSON, model, and publication-figure exports;
- persistent CIFAR-100 storage in Google Drive;
- RAM-safe `uint8` dataset storage, bounded TensorFlow prefetching, and explicit cleanup between methods and seeds.

## Important interpretation

This code evaluates empirical unlearning. It does not provide a formal privacy certificate. Forgotten training data are never used by either ScrubRef variant, but they are used by evaluation routines and by baselines whose published access models require them.


## 1. Environment and Experiment Setup
This section initializes the execution environment, verifies runtime dependencies,
prepares persistent storage on Google Drive, downloads or links the dataset, and loads the experiment configuration.

### 1.1 Google Colab Setup

The following cell mounts Google Drive and prepares the repository for
execution in Colab. Skip this cell when running the notebook locally.


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys


os.chdir("/content")
drive.mount("/content/drive")

GITHUB_REPO = "https://github.com/imnima2005/scrubref-unlearning.git"
REPO_DIR_NAME = "scrubref-unlearning"

DRIVE_ROOT = Path("/content/drive/MyDrive")
REPO_DIR = DRIVE_ROOT / REPO_DIR_NAME

if not REPO_DIR.exists():
    print("Cloning repository...")
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO, str(REPO_DIR)],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print("Error:")
        print(result.stderr)
        raise RuntimeError("Clone failed")
    print("Clone successful!")
else:
    print(f"already exist: {REPO_DIR}")

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Project root:", Path.cwd())

### 1.2 Runtime Diagnostics

Reports the Python version, platform, and installed package versions without
modifying or installing any packages. Also checks for available GPU devices.


In [ ]:
# Colab environment check — do not install or downgrade packages here.
import importlib.metadata as metadata
import platform
import sys

PACKAGES = (
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "tensorflow",
    "joblib",
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

for package in PACKAGES:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "not installed"

    print(f"{package}: {version}")

print("Environment check complete.")

In [ ]:
from __future__ import annotations

import ctypes
import gc
import json
import math
import os
import random
import shutil
import tarfile
import time
import urllib.request
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple, Union

try:
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import tensorflow as tf
except ValueError as exc:
    if "numpy.dtype size changed" in str(exc):
        raise RuntimeError(
            "The Colab VM contains an incompatible NumPy/Pandas binary stack. "
            "Use Runtime > Disconnect and delete runtime, reconnect, and run this notebook "
            "without executing any pip-install cell."
        ) from exc
    raise
from sklearn import set_config as sklearn_set_config
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

sklearn_set_config(working_memory=256)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


### 1.3 Persistent Keras Cache

Creates a symbolic link to store Keras datasets permanently in Google Drive,
preventing re-downloading across sessions.


In [ ]:
import shutil
from pathlib import Path
import os

# Configure persistent Keras datasets directory directly in your project's data folder
DRIVE_DATASETS = REPO_DIR / "data"
LOCAL_DATASETS = Path.home() / ".keras" / "datasets"

DRIVE_DATASETS.mkdir(parents=True, exist_ok=True)
LOCAL_DATASETS.parent.mkdir(parents=True, exist_ok=True)

# Replace only the ephemeral local cache directory with a Drive-backed symlink
if LOCAL_DATASETS.is_symlink():
    if LOCAL_DATASETS.resolve() != DRIVE_DATASETS.resolve():
        LOCAL_DATASETS.unlink()
elif LOCAL_DATASETS.exists():
    shutil.rmtree(LOCAL_DATASETS)

if not LOCAL_DATASETS.exists():
    os.symlink(str(DRIVE_DATASETS), str(LOCAL_DATASETS), target_is_directory=True)

os.environ["KERAS_HOME"] = str(REPO_DIR)

print("Persistent Keras cache:", DRIVE_DATASETS)
print("Local cache link:", LOCAL_DATASETS, "->", LOCAL_DATASETS.resolve())

### 1.4 Dataset Preparation

Checks, downloads from verified mirrors, verifies via MD5 checksum, and extracts
the dataset into the persistent Google Drive folder. Skips download if the dataset
is already complete.


In [ ]:
# Prepare CIFAR-100 from a fast mirror and keep it permanently in Google Drive.
from pathlib import Path
import hashlib
import shutil
import subprocess
import tarfile


def file_md5(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


# Use the same DRIVE_DATASETS path (REPO_DIR / "data") configured in the symlink cell
main_dataset_dir = REPO_DIR / "data"
dataset_dir = main_dataset_dir / "cifar100"
dataset_dir.mkdir(parents=True, exist_ok=True)

local_archive = Path("/content/cifar-100-python.tar.gz")
drive_archive = dataset_dir / "cifar-100-python.tar.gz"
extracted_dir = dataset_dir / "cifar-100-python"
expected_md5 = "eb9058c3a382ffc7106e4002c42a8d85"

mirror_urls = [
    "https://huggingface.co/datasets/nakroy/cifar100-python/resolve/main/cifar-100-python.tar.gz",
    "https://data.brainchip.com/dataset-mirror/cifar100/cifar-100-python.tar.gz",
]

required_extracted_files = [
    extracted_dir / "train",
    extracted_dir / "test",
    extracted_dir / "meta",
]

if all(path.exists() for path in required_extracted_files):
    print("CIFAR-100 already prepared:", extracted_dir)
else:
    if drive_archive.exists() and file_md5(drive_archive) != expected_md5:
        print("Removing invalid Drive archive:", drive_archive)
        drive_archive.unlink()

    if not drive_archive.exists():
        local_archive.unlink(missing_ok=True)
        downloaded = False
        for url in mirror_urls:
            print("Trying:", url)
            result = subprocess.run(
                [
                    "wget",
                    "-c",
                    "--progress=bar:force:noscroll",
                    "--timeout=30",
                    "--tries=3",
                    "-O",
                    str(local_archive),
                    url,
                ],
                check=False,
            )
            if result.returncode != 0 or not local_archive.exists():
                continue
            actual_md5 = file_md5(local_archive)
            print("MD5:", actual_md5)
            if actual_md5 == expected_md5:
                shutil.copy2(local_archive, drive_archive)
                downloaded = True
                break
            local_archive.unlink(missing_ok=True)

        if not downloaded:
            raise RuntimeError("All CIFAR-100 mirrors failed or returned an invalid archive.")

    print("Extracting CIFAR-100 into Google Drive...")
    with tarfile.open(drive_archive, "r:gz") as archive:
        try:
            archive.extractall(dataset_dir, filter="data")
        except TypeError:
            archive.extractall(dataset_dir)

    missing = [str(path) for path in required_extracted_files if not path.exists()]
    if missing:
        raise RuntimeError("CIFAR-100 extraction is incomplete:\n" + "\n".join(missing))
    print("Dataset prepared:", extracted_dir)

local_archive.unlink(missing_ok=True)

### 1.5 Experiment Configuration

Loads the YAML experiment configuration and applies settings such as the run mode
(e.g., debug vs. full execution).


In [ ]:
from src.config import ExperimentConfig
from src import config as config_module
from src import data_loader as data_loader_module

CFG = ExperimentConfig.from_yaml(REPO_DIR / "configs"/ "publication" / "cifar10.yaml")
CFG.run_mode = "debug"
CFG.apply_mode()

# Make CFG available inside the source modules
config_module.CFG = CFG
data_loader_module.CFG = CFG

# Make other notebook-level variables available if needed
data_loader_module.IN_COLAB = True
data_loader_module.REPO_DIR = REPO_DIR

print("CFG injected successfully")
print("run_mode:", CFG.run_mode)
print("dataset:", CFG.dataset)

### 1.6 Reproducibility and Runtime Initialization

This step initializes the experiment's reproducibility settings before any data processing or model training begins. The random seed is read from the active experiment configuration and applied globally to reduce run-to-run variation.

Reusable utilities for setting determinism and managing TensorFlow memory are implemented in `src/utils.py`. The notebook only imports and invokes the required utility.


In [ ]:
from src.utils import set_global_determinism

set_global_determinism(CFG.seeds[0])

## 2. Dataset Preparation and Unlearning Split

This section loads the configured dataset, creates a reproducible retain/forget split, and prepares the TensorFlow input pipelines used by training and evaluation.

Dataset-specific loading, split construction, image decoding, augmentation, and `tf.data` pipeline logic are implemented in `src/data_loader.py`.

### 2.1 Dataset Loading and Validation

Load the dataset specified by the experiment configuration and validate that the expected files and class structure are available. The loader returns a `DataBundle` containing the training and test examples, labels, class names, and image metadata.


In [ ]:
from src.data_loader import (
    DataBundle,
    ensure_imagewoof,
    load_dataset,
    list_directory_split,
    load_directory_bundle
)

### 2.2 Reproducible Retain/Forget Split

Partition the training and test data into retain and forget subsets according to the configured forget classes. A validation subset is sampled from the retain-training data using the experiment seed, so the split can be reproduced across runs.


In [ ]:
from src.data_loader import (
    UnlearningSplit,
    make_unlearning_split,
)

### 2.3 Input Pipeline and Augmentation

Build TensorFlow datasets for training and evaluation. Images are decoded or resized as needed, optional training augmentation is applied, and batching, shuffling, determinism, and prefetching are configured through the experiment settings.


In [ ]:
from src.data_loader import (
    decode_item,
    augment_image,
    make_dataset,
    materialize_items,
)

## 3. Model and Unlearning Method

This section defines the model architecture and the training utilities used to prepare the original model and support subsequent unlearning experiments. The reusable implementation is organized in `src/models.py` and `src/training.py`, while the notebook documents the experimental setup and orchestrates model loading or training.


### 3.1 Model Architecture and Model Utilities

Define the model architecture and reusable model-inspection utilities. These functions support model construction, checkpoint cloning, output interpretation, feature extraction, and selective control of trainable layers.


In [ ]:
from src.models import (
    build_resnet18,
    classifier_layer,
    clone_architecture,
    clone_with_weights,
    fresh_clone,
    leaf_layers,
    model_containers,
    output_to_logits,
    output_to_probs,
    set_all_trainable,
    set_tail_trainable,
)

### 3.2 Training and Original Model Initialization

Load a compatible pretrained checkpoint when available; otherwise, initialize and train the original model using the configured dataset and training parameters. The original model serves as the reference point for the subsequent unlearning procedure.


In [ ]:
from src.training import (
    compile_supervised,
    infer_output_is_logits,
    train_supervised,
    load_or_train_original,
)

### 3.3 Synthetic Proxy Data Generation

Generate class-conditioned synthetic proxy images by optimizing input images against the frozen teacher model. The target classes are assigned in a balanced schedule, and the resulting images, labels, and generation metadata are cached for reuse.

The proxy-generation procedure is implemented in `src/proxy_generation.py`. The notebook records the configuration and invokes the generator.


In [ ]:
from src.proxy_generation import (
    balanced_proxy_targets,
    generate_class_proxies,
    generate_unsir_error_maximizing_noise
)

### 3.4 ScrubRef Unlearning Procedure

Apply the ScrubRef unlearning procedure to the original model using the retain split and class-conditioned proxy samples. The student is initialized from the original model and optimized to preserve utility on retained data while updating its predictions—and optionally its representations—on proxy samples associated with the forget classes.

The implementation is provided in `src/scrubref.py`. The notebook records the configuration and invokes the procedure; evaluation is reported separately in Section 4.


In [ ]:
from src.scrubref import (
    scrubref_train
)

### 3.5 Baseline Unlearning Methods

Implement comparison methods used to assess the ScrubRef procedure. Each baseline starts from the same original model and uses the same unlearning split and evaluation protocol unless its method-specific definition requires otherwise. Baseline implementations are organized under `src/baselines/`; the notebook configures and runs the selected methods.


In [ ]:
from src.baselines.simple import (
    logit_clamp_baseline,
    head_reset_baseline,
    retain_finetune_baseline,
    retrain_baseline,
)

from src.baselines.distillation import (
    bad_teacher_baseline,
    scrub_baseline
)

from src.baselines.ssd import (
    ssd_baseline,
    estimate_fisher_diagonal
)

from src.baselines.unsir import (
    unsir_baseline
)


## 4. Evaluation and Unlearning Assessment

Evaluate the utility, calibration, privacy-related behavior, and parameter changes of the trained or unlearned model. Reusable evaluation and metric implementations are defined in `src/metrics.py`; the notebook calls them and presents the resulting metrics.

Utility metrics measure predictive performance on retained and forgotten classes. Membership inference metrics assess whether forget-set training examples can be distinguished from forget-set test examples. Parameter-change statistics quantify how much the model changed during unlearning.


### 4.1 Prediction and Utility Metrics

Generate class probabilities and compute predictive performance metrics, including overall, retained, and forgotten accuracy, top-5 accuracy, macro-F1, balanced accuracy, negative log-likelihood, and per-class accuracy.


In [ ]:
from src.metrics import evaluate_utility, per_class_accuracy, predict_probabilities

### 4.2 Calibration Metrics

Measure the quality of the model's predicted probabilities using Expected Calibration Error (ECE) and the multiclass Brier score. These complement accuracy by evaluating confidence calibration.


In [ ]:
from src.metrics import expected_calibration_error, brier_multiclass

### 4.3 Membership Inference Evaluation

Evaluate membership inference on forgotten examples by comparing forget-set training members with forget-set test non-members. Report AUC-based scores from confidence, loss, entropy, modified entropy, and a learned attack, along with the learned attack's average precision.


In [ ]:
from src.metrics import entropy_scores, safe_auc, learned_mia, evaluate_membership_inference, modified_entropy_scores

### 4.4 Parameter and Runtime Statistics

Quantify the model's parameter changes relative to its original version and, when available, record peak GPU memory usage. These statistics help characterize the unlearning procedure's computational and model-level effects.


In [ ]:
from src.metrics import parameter_change_statistics, peak_gpu_memory_mb

## 5. Experiment Pipeline Components

This section defines the reusable components used by the experiment pipeline, including experiment configuration, method orchestration, evaluation, and artifact generation. The notebook invokes the completed pipeline in Section 8; implementation details are organized in `src/experiment_runner.py`.



### 5.1 Seed-Level Experiment Runner

Define the workflow for one dataset and random seed, including split construction, original-model loading, proxy generation, method execution, evaluation, and artifact storage. The runner is invoked later in Section 8.


In [ ]:
from src.experiment_runner import (
    build_method,
    method_model_path,
    load_saved_model,
    run_seed,
    save_json,
)

## 6. Representation Audit and Analysis

Import representation auditing utilities from `src.representation_audit` to compute linear CKA, Jensen-Shannon divergence on output probability distributions, and linear probe accuracy on retained vs. forgotten classes.


### 6.1 Representation Audit

Extract and cache train, test, and CKA features for the configured methods. Compare each method with the original model and, when available, the retrained reference using linear CKA, representation probes, predictive-distribution divergence, and prediction disagreement. The audit implementation is kept in the experiment-analysis modules; the notebook launches it and inspects the resulting metrics.


In [ ]:
from src.representation_audit import (
    stratified_indices,
    extract_features,
    linear_cka,
    representation_probe_metrics,
    run_representation_audit
)

### 6.2 Per-Seed Figures

Visualize the per-seed results for retained and forgotten accuracy, membership-inference AUC, calibration, and representation similarity. Figures are saved in the run's `figures/` directory and provide a visual summary of the corresponding metric CSV files.


In [ ]:
from src.representation_audit import (
    make_seed_figures,
)

## 7. Runtime and Dataset Preflight

Verify that the dataset can be loaded with the expected image data types and estimate its memory footprint before starting the experiments. Release the temporary dataset bundle after the check.


In [ ]:
from src.utils import array_memory_mb, release_memory

PREFLIGHT_BUNDLE = load_dataset(CFG)
print("Train dtype:", PREFLIGHT_BUNDLE.train_items.dtype)
print("Test dtype:", PREFLIGHT_BUNDLE.test_items.dtype)
print(
    "Dataset image RAM:",
    f"{array_memory_mb(PREFLIGHT_BUNDLE.train_items, PREFLIGHT_BUNDLE.test_items):.2f} MB",
)
if not PREFLIGHT_BUNDLE.path_mode and PREFLIGHT_BUNDLE.train_items.dtype != np.uint8:
    raise RuntimeError("RAM preflight failed: CIFAR images are not uint8.")
del PREFLIGHT_BUNDLE
release_memory(clear_session=False)
print("RAM preflight passed.")


In [ ]:
import inspect
print(inspect.getsource(load_or_train_original))

In [ ]:
CFG.enabled_methods = [
    "original", "retrained", "logit_clamp",
    "head_reset", "retain_finetune"
    # "scrubref_head"  <-- Temporarily comment out or delete this section.
]

In [ ]:
import os
import pandas as pd
csv_path = Path(CFG.results_root) / CFG.dataset / f"seed_{42}" / "utility_metrics.csv"

#csv_path = "/content/drive/MyDrive/scrubref_revision_results/cifar100/seed_42/utility_metrics.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    cols = ["method", "overall_accuracy", "retained_accuracy", "forgotten_accuracy", "macro_f1", "nll"]
    available_cols = [c for c in cols if c in df.columns]
    display(df[available_cols])
else:
    print(f"⚠️ Result file not found at: {csv_path}")
    print("Please execute the experiment runner cell first (under 'Execute experiments') to generate the results.")

## 8. Execute Experiments

Run the experiment pipeline for each configured seed. Existing results may be reused when `FORCE_RERUN` is `False`; set it to `True` to force the pipeline to rerun instead of relying on cached results.


In [ ]:
FORCE_RERUN = False

SEED_OUTPUTS = []
for experiment_seed in CFG.seeds:
    print(f"\n{'=' * 90}\nRunning {CFG.dataset}, seed={experiment_seed}\n{'=' * 90}")
    SEED_OUTPUTS.append(run_seed(CFG, experiment_seed, force_rerun=FORCE_RERUN))

print("\n" + "="*90 + "\nAll experiments completed successfully!\n" + "="*90)
SEED_OUTPUTS

## 9. Aggregate all seeds

The aggregation cell produces mean, standard deviation, and 95% normal-approximation confidence intervals for every numeric metric.

In [ ]:
def aggregate_metric_file(cfg: ExperimentConfig, filename: str, output_name: str) -> pd.DataFrame:
    frames = []
    for seed in cfg.seeds:
        path = Path(cfg.results_root) / cfg.dataset / f"seed_{seed}" / filename
        if path.exists():
            frames.append(pd.read_csv(path))
    if not frames:
        raise FileNotFoundError(f"No {filename} files were found.")
    combined = pd.concat(frames, ignore_index=True)
    numeric_columns = [column for column in combined.select_dtypes(include=[np.number]).columns if column != "seed"]
    grouped = combined.groupby("method")[numeric_columns].agg(["mean", "std", "count"])
    rows = []
    for method in grouped.index:
        row: Dict[str, object] = {"method": method}
        for metric in numeric_columns:
            mean = grouped.loc[method, (metric, "mean")]
            std = grouped.loc[method, (metric, "std")]
            count = grouped.loc[method, (metric, "count")]
            ci95 = 1.96 * std / math.sqrt(count) if count > 1 and np.isfinite(std) else float("nan")
            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
            row[f"{metric}_ci95"] = ci95
        rows.append(row)
    output = pd.DataFrame(rows)
    aggregate_dir = Path(cfg.results_root) / cfg.dataset / "aggregate"
    aggregate_dir.mkdir(parents=True, exist_ok=True)
    combined.to_csv(aggregate_dir / f"all_seeds_{filename}", index=False)
    output.to_csv(aggregate_dir / output_name, index=False)
    return output


AGGREGATED_UTILITY = aggregate_metric_file(CFG, "utility_metrics.csv", "utility_summary.csv")
AGGREGATED_MIA = aggregate_metric_file(CFG, "mia_metrics.csv", "mia_summary.csv")
AGGREGATED_RUNTIME = aggregate_metric_file(CFG, "runtime_metrics.csv", "runtime_summary.csv")
AGGREGATED_REPRESENTATION = aggregate_metric_file(CFG, "representation_metrics.csv", "representation_summary.csv")

print("Utility summary")
display(AGGREGATED_UTILITY)
print("MIA summary")
display(AGGREGATED_MIA)
print("Representation summary")
display(AGGREGATED_REPRESENTATION)

## 10. Reviewer-focused result checks

These checks do not decide acceptance. They identify invalid or weak result patterns before values are copied into the manuscript.

In [ ]:
def reviewer_readiness_checks(cfg: ExperimentConfig) -> pd.DataFrame:
    aggregate_dir = Path(cfg.results_root) / cfg.dataset / "aggregate"
    utility = pd.read_csv(aggregate_dir / "utility_summary.csv").set_index("method")
    mia = pd.read_csv(aggregate_dir / "mia_summary.csv").set_index("method")
    representation = pd.read_csv(aggregate_dir / "representation_summary.csv").set_index("method")
    rows = []
    for method in ["scrubref_head", "scrubref_rep"]:
        if method not in utility.index:
            continue
        retain_drop = utility.loc["original", "retained_accuracy_mean"] - utility.loc[method, "retained_accuracy_mean"]
        forgotten_accuracy = utility.loc[method, "forgotten_accuracy_mean"]
        learned_mia = mia.loc[method, "mia_learned_auc_mean"]
        cka_original = representation.loc[method, "cka_to_original_mean"]
        cka_retrained = representation.loc[method, "cka_to_retrained_mean"]
        linear_probe_forget = representation.loc[method, "linear_probe_forgotten_accuracy_mean"]
        rows.append({
            "method": method,
            "retained_accuracy_drop": retain_drop,
            "forgotten_accuracy": forgotten_accuracy,
            "learned_mia_auc": learned_mia,
            "cka_to_original": cka_original,
            "cka_to_retrained": cka_retrained,
            "closer_to_retrained_than_original": bool(cka_retrained > cka_original),
            "linear_probe_forgotten_accuracy": linear_probe_forget,
            "output_forgetting_pass": bool(forgotten_accuracy <= 0.05),
            "empirical_mia_pass": bool(learned_mia <= 0.60),
            "utility_pass": bool(retain_drop <= 0.03),
        })
    checks = pd.DataFrame(rows)
    checks.to_csv(aggregate_dir / "reviewer_readiness_checks.csv", index=False)
    return checks


REVIEWER_CHECKS = reviewer_readiness_checks(CFG)
display(REVIEWER_CHECKS)

## Optional ablation runner

The main result must be compared with mechanisms that can trivially suppress outputs. This ablation executes the two ScrubRef variants under four settings:

- no logit surgery, no clamp;
- logit surgery only;
- clamp only;
- both mechanisms.

In [ ]:
def run_scrubref_ablations(base_cfg: ExperimentConfig, seed: int) -> pd.DataFrame:
    rows = []
    settings = [
        (False, False, "none"),
        (True, False, "logit_surgery"),
        (False, True, "confidence_clamp"),
        (True, True, "surgery_and_clamp"),
    ]
    for use_surgery, use_clamp, label in settings:
        ablation_cfg = ExperimentConfig(**asdict(base_cfg))
        ablation_cfg.seeds = (seed,)
        ablation_cfg.enabled_methods = ("scrubref_head", "scrubref_rep")
        ablation_cfg.representation_methods = ("original", "retrained", "scrubref_head", "scrubref_rep")
        ablation_cfg.enable_main_method_logit_surgery = use_surgery
        ablation_cfg.enable_main_method_clamp = use_clamp
        ablation_cfg.results_root = str(Path(base_cfg.results_root) / "ablations" / label)
        outputs = run_seed(ablation_cfg, seed, force_rerun=False)
        utility = pd.read_csv(outputs["utility"])
        utility["ablation"] = label
        rows.append(utility)
    result = pd.concat(rows, ignore_index=True)
    output_dir = Path(base_cfg.results_root) / "ablations"
    output_dir.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_dir / f"{base_cfg.dataset}_ablation_results.csv", index=False)
    return result


RUN_ABLATIONS = True
ABLATION_RESULTS = run_scrubref_ablations(CFG, CFG.seeds[0]) if RUN_ABLATIONS else pd.DataFrame()
display(ABLATION_RESULTS)

## Output package

After completion, the aggregate directory contains:

- `utility_summary.csv`
- `mia_summary.csv`
- `runtime_summary.csv`
- `representation_summary.csv`
- `reviewer_readiness_checks.csv`
- all per-seed result files, model checkpoints, proxy arrays, and figures

Use only values generated by these files in the revised manuscript. Do not reuse numerical results from the submitted notebook because its scrub branch and membership evaluation were invalid.

## Method-label discipline

- `scrubref_head` is the corrected original algorithmic scope.
- `scrubref_rep` is an algorithmic extension and must be described as such.
- `unsir`, `bad_teacher`, `scrub`, and `ssd` are TensorFlow-native adaptations. Before publication, compare their settings and outputs against the official repositories and state any implementation differences.
- Membership-inference values are empirical audit results, not formal privacy guarantees.

## RAM-safe execution notes

- Use **Runtime → Run all** after a clean Colab runtime is connected.
- Keep `FORCE_RERUN = False` to resume completed methods.
- Do not convert complete CIFAR arrays to `float32`; conversion occurs per mini-batch.
- The publication configuration uses three seeds and all enabled methods.
- If Colab disconnects, reconnect, run from the top, and completed model checkpoints will be reused.
